<h2>Food (v0.03) - 'Food (v0.03).ipynb'</h2>

<u><h5>Context</h5></u>

The notebook aims to isolate NZF impacts modelled for individual commodities with 'CPC' codes forwarded over by David Forengie on 23rd September 2026.

The objectives of the notebook are:

<ol>
    <li>
        To develop the representation of CPC codes provided by David, as its not completely clear how these work (e.g. where some CPC codes are only 3 digits, whereas others are 5). 
    </li>
    <li>
        To read-in 'v0.64' of the 'Impact' analysis results for the country in question, then isolate these records for the CPC codes identified in the step above.
    </li>
</ol>

<u><h5>Version Documentation</h5></u>

'v0.01' outlined an initial method for completing the process. 'v0.02' refined the method and developed a fully functioning notebook, utilising results from the 'Impact' notebooks which now include CPC codes as fields in the output data tables. Upon inspection, it was found that the identification of CPC codes wasn't functioning properly, where CPC codes specified by David wouldn't show in the dataset, yet HSa4 codes given in the CPC manual would show. This likely explains much of the losses observed when pulling by CPC codes, so 'v0.03' utilises a new version of the codes sent through David which have been mapped to the HS system ahead of time. 

In [1]:
# Import packages
import os
import base64
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 250)
pd.set_option('display.max_rows', 250)
pd.options.mode.chained_assignment = None
from IPython.display import HTML
import matplotlib.pyplot as plt
import matplotlib.style
plt.style.use('seaborn-v0_8')

def download_link(df, title="Download CSV file", filename="data.csv"):
    csv = df.to_csv()
    b64 = base64.b64encode(csv.encode())
    payload = b64.decode()
    html = '<a download="{filename}" href="data:text/csv;base64,{payload}" target="_blank">{title}</a>'
    html = html.format(payload=payload,title=title,filename=filename)
    return HTML(html)

# define function for counting values in each data field
trade_im_field_count = lambda field: trade_im[field].value_counts().to_frame().reset_index()
trade_ex_field_count = lambda field: trade_ex[field].value_counts().to_frame().reset_index()

version = "v0.03"
input_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/"
output_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/food/"

# 1. Set up the notebook

## 1.1 Let's set the country we're interested in and set up a results folder.

In [2]:
# Select one of our countries to work on...
iso_code = "882"
iso_country = "Samoa"
year = 2025
import_portfolio = "h5_ag6_by_sea_M*.csv" # Imports, '_M' | Foreign Imports, '_FM' | Re-Imports, '_RM'
export_portfolio = "h5_ag6_by_sea_X*.csv" # Exports, '_X' | Domestic Exports, '_DX' | Re-exports, '_RX'
gdp = 1.290*1e9

# Create output Journey directory folder if necessary
os.makedirs(output_dir+"{0}/".format(iso_country)) if not os.path.exists(output_dir+"{0}/".format(iso_country)) else "Output directory already exists."

## 1.2 Import and Format 'Impact Analysis' Results

In [3]:
impact_im_c = [
    "trade_idx", "reporterCode", "reporterDesc", "partnerCode", "partnerDesc", "flowDesc", "motDesc", 
    "HS6", "HS6_desc", "HS4", "HS4_desc", "HS2", "HS2_desc", "cpc", "cpc_desc", "eora_desc", "gtap", "gtap_desc", 
    "cifvalue", "fobvalue", "primaryValue", "qtyUnitAbbr", "qty", "netWgt", "vol_kg_vtw", 
    "sea_dist_km", "sea_dist_nm", "tnm", "fac_vessel_types", "fac_vessel_types_str", 
    #"TandT_match_desc", "TandT_ad_val_pct", "TandT_N", "TandT_ciffob", "TandT_cifvalue", "TandT_fobvalue", "cost_int_match_desc",
    "bau_usd_26", "levy_usd_26", "asis_usd_26", "brazil_usd_26", "liscr_usd_26", "direct_usd_26", 
    "bau_usd_29", "levy_usd_29", "asis_usd_29", "brazil_usd_29", "liscr_usd_29", "direct_usd_29", 
    "bau_usd_32", "levy_usd_32", "asis_usd_32", "brazil_usd_32", "liscr_usd_32", "direct_usd_32", 
    "bau_usd_35", "levy_usd_35", "asis_usd_35", "brazil_usd_35", "liscr_usd_35", "direct_usd_35", 
    "bau_usd_38", "levy_usd_38", "asis_usd_38", "brazil_usd_38", "liscr_usd_38", "direct_usd_38", 
    "bau_usd_41", "levy_usd_41", "asis_usd_41", "brazil_usd_41", "liscr_usd_41", "direct_usd_41", 
    "bau_usd_44", "levy_usd_44", "asis_usd_44", "brazil_usd_44", "liscr_usd_44", "direct_usd_44", 
    "bau_usd_47", "levy_usd_47", "asis_usd_47", "brazil_usd_47", "liscr_usd_47", "direct_usd_47", 
    "bau_usd_50", "levy_usd_50", "asis_usd_50", "brazil_usd_50", "liscr_usd_50", "direct_usd_50"	
]

impact_im_d = {
    "reporterCode":"str", "partnerCode":"str", 
    "HS6":"str", "HS4":"str", "HS2":"str", "HS2":"str", "gtap":"str", "cpc":"str"
}

impact_im = pd.read_csv(
    input_dir + "impact/{0}/impact_im_v0.64.csv".format(iso_country), 
    dtype=impact_im_d,
    usecols=impact_im_c
)[impact_im_c]

print("Aggregate trade portfolio statistics for '{0}': \n".format(iso_country))
print("Number of trade records present in dataset: {0}".format(impact_im.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(impact_im.partnerCode.unique())))
print("Modes of transport present in dataset: {0}".format(impact_im.motDesc.unique()))
print("Total value within the dataset: US${0}m".format(np.round(impact_im.cifvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(impact_im.vol_kg_vtw.sum() / 1e6, 1)))

impact_im["cpc_cl"] = [str(i).split('.')[0] for i in impact_im.cpc.values]
impact_im = impact_im.drop(columns="trade_idx").reset_index(drop=True).reset_index().rename(columns={"index":"trade_idx"})
display(impact_im.head(2))

Aggregate trade portfolio statistics for 'Samoa': 

Number of trade records present in dataset: 11218
Number of partner countries present in dataset: 79
Modes of transport present in dataset: ['Sea']
Total value within the dataset: US$413.7m
Total weight within the dataset: 451.3 million kilograms


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl
0,0,882,Samoa,554,New Zealand,Import,Sea,40690,040690 | Dairy produce; chees,0406,0406 | Cheese and curd,04,04 | Dairy produce; birds,2295,Cheese and curd,Agriculture,22,MIL - Dairy products,39813.073551,NaN,39813.073551,kg,9178.55908,9178.55908,8153.717463,2813.0,1747.916623,14252.018293,['Container'],['Container'],43.833339,43.748538,43.756967,43.756967,43.798400,43.817147,45.440501,75.782749,48.460043,45.654378,45.801012,47.236954,44.617395,73.542473,50.724276,47.515919,46.615835,48.467961,43.899522,81.151083,60.833487,55.414975,51.897461,50.241822,43.196277,87.826141,70.870445,66.644570,54.813267,53.401706,42.592554,89.466089,84.610828,87.042609,59.057668,55.859505,41.739253,85.511644,82.867310,88.183568,69.791232,56.569521,41.055409,77.682360,80.210559,87.531271,68.394472,57.188444,40.362654,71.278985,73.045706,84.404435,69.641151,57.623039,2295
1,1,882,Samoa,554,New Zealand,Import,Sea,50400,050400 | Animal products; gut,0504,"0504 | Guts, bladders and s",05,05 | Animal originated pr,3911,"Raw offal, inedible (including pigs' bristles,...",Agriculture,10,OAP - Animal products n.e.c.,4805.128484,NaN,4805.128484,kg,99.80000,99.80000,1130.646732,2813.0,1747.916623,1976.276218,['Container'],['Container'],6.078212,6.066453,6.067622,6.067622,6.073367,6.075967,6.301071,10.508522,6.719780,6.330729,6.351062,6.550179,6.186934,10.197871,7.033753,6.588862,6.464051,6.720878,6.087389,11.252929,8.435561,7.684196,7.196435,6.966853,5.989873,12.178536,9.827350,9.241363,7.600759,7.405023,5.906157,12.405941,11.732680,12.069886,8.189315,7.745837,5.787832,11.857593,11.490912,12.228099,9.677700,7.844292,5.693006,10.771934,11.122510,12.137647,9.484016,7.930116,5.596944,9.884001,10.128986,11.704060,9.656888,7.990380,3911


# 2. Subset the 'Impact Analysis' results for food security-related CPC codes

## 2.1 Let's get a sense of how much of import value is associated with food-relevant goods across classification systems

### 2.1.1 Harmonised System (HS01-23)

In [4]:
#print("HS groupings: \n\n{0}\n".format(pd.Series(impact_im.HS2_desc.unique()).sort_values().values))
HSa2_good_groups = [
    "01", "02", "03", "04", "05", "07", "08", # "06", 
    "09", "10", "11", "12", "13", "14", "15", "16", 
    "17", "18", "19", "20", "21", "22", "23"
]

impact_im_HSa2_food = impact_im[(impact_im.HS2.isin(HSa2_good_groups))]
impact_im_HSa2_food_g = impact_im_HSa2_food.groupby(["HS2_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_HSa2_food_g["pct_val"] = round(100 * impact_im_HSa2_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_HSa2_food_g["pct_vol"] = round(100 * impact_im_HSa2_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_HSa2_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_HSa2_food_g.round(1)
))

Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): $121.9m (29.5%), ... 

                             cifvalue  vol_kg_vtw  pct_val  pct_vol
HS2_desc                                                           
02 | Meat and edible meat  22866497.2  11353710.7      5.5      2.5
19 | Preparations of cere  16663544.7   7486125.9      4.0      1.7
04 | Dairy produce; birds  15980127.0  11608233.6      3.9      2.6
16 | Meat, fish, crustace   9866186.2   2131340.0      2.4      0.5
21 | Miscellaneous edible   9310346.9   3020093.3      2.3      0.7
11 | Products of the mill   7350788.3  16968503.9      1.8      3.8
22 | Beverages, spirits a   6880507.7   7824895.2      1.7      1.7
10 | Cereals                6607046.3  11551724.0      1.6      2.6
17 | Sugars and sugar con   6022157.3  11010798.2      1.5      2.4
20 | Preparations of vege   5082121.5   3809457.5      1.2      0.8
15 | Animal, vegetable or   4202255.0   2538666.5      1.0      0.6
07 | Vegetables

### 2.1.2 EORA groupings ('Food & Beverages' and 'Fishing')

In [5]:
print("EORA groupings: \n\n{0}\n".format(pd.Series(impact_im.eora_desc.unique()).sort_values().values))
eora_good_groups = ["Food & Beverages", "Agriculture", "Fishing"]

impact_im_eora_food = impact_im[(impact_im.eora_desc.isin(eora_good_groups))]
impact_im_eora_food_g = impact_im_eora_food.groupby(["eora_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_eora_food_g["pct_val"] = round(100 * impact_im_eora_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_eora_food_g["pct_vol"] = round(100 * impact_im_eora_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_eora_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_eora_food_g.round(1)
))

EORA groupings: 

['Agriculture' 'Electrical and Machinery' 'Fishing' 'Food & Beverages'
 'Metal Products' 'Mining and Quarrying' 'Other Manufacturing'
 'Petroleum, Chemical and Non-Metallic Mineral Products'
 'Textiles and Wearing Apparel' 'Transport Equipment' 'Wood and Paper' nan]

Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): $118.8m (28.7%), ... 

                    cifvalue  vol_kg_vtw  pct_val  pct_vol
eora_desc                                                 
Agriculture       62353255.8  60191660.8     15.1     13.3
Food & Beverages  56326959.8  33363315.0     13.6      7.4
Fishing             100915.9     21977.7      0.0      0.0



### 2.1.3 GTAP (16x headings)

In [6]:
#print("GTAP groupings: \n\n{0}\n".format(pd.Series(impact_im.gtap_desc.unique()).sort_values().values))
gtap_food_groups = [
    "GRO - Cereal grains n.e.c.", "PDR - Paddy rice", "PCR - Processed rice", "SGR - Sugar", 
    "CMT - Bovine meat prods", "CTL - Bovine cattle, sheep and goats, horses", "FSH - Fishing", "OMT - Meat products n.e.c.",
    "MIL - Dairy products", "OFD - Food products n.e.c.", "V_F - Vegetables, fruit, nuts", "OSD - Oil seeds", 
    "OCR - Crops n.e.c.", "PFB - Plant-based fibers", "VOL - Vegetable oils and fats", 
    "OAP - Animal products n.e.c."
]

impact_im_gtap_food = impact_im[(impact_im.gtap_desc.isin(gtap_food_groups))]
impact_im_gtap_food_g = impact_im_gtap_food.groupby(["gtap_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_gtap_food_g["pct_val"] = round(100 * impact_im_gtap_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_gtap_food_g["pct_vol"] = round(100 * impact_im_gtap_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via GTAP: ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_gtap_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_gtap_food_g.head(15).round(1)
))

Value of food imports identified via GTAP: $109.6m (26.5%), ... 

                                 cifvalue  vol_kg_vtw  pct_val  pct_vol
gtap_desc                                                              
OFD - Food products n.e.c.     49132135.3  33031529.6     11.9      7.3
OMT - Meat products n.e.c.     22329334.5  11268208.4      5.4      2.5
MIL - Dairy products           16246073.6  11095324.8      3.9      2.5
PCR - Processed rice            6555226.7  11386428.8      1.6      2.5
V_F - Vegetables, fruit, nuts   4720224.4   6667393.8      1.1      1.5
VOL - Vegetable oils and fats   4198165.8   2538469.5      1.0      0.6
SGR - Sugar                     3031459.2   6638229.7      0.7      1.5
CMT - Bovine meat prods         2825765.8    516634.2      0.7      0.1
OCR - Crops n.e.c.               359601.5    145126.3      0.1      0.0
PFB - Plant-based fibers          88897.0    104066.4      0.0      0.0
OAP - Animal products n.e.c.      50083.1     11854.2      0.0      0.

## 2.2 Comparison of results

In [7]:
print("Share of total import value associated with food-relevant goods by classification system...\n\nHSa2 {0}% \nGTAP {1}% \nEORA {2}%".format(
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))

Share of total import value associated with food-relevant goods by classification system...

HSa2 29.5% 
GTAP 26.5% 
EORA 28.7%


# 3. Take HSa4 codes associated with specific CPCs

## 3.1 Set HSa4 Codes we're interested in

In [8]:
hsa4_codes_food = {
    "Dairy": [
        "0401", "0402", "0403", "0404", "0405", "0406",
    ],
    "Fats and Oils": [
        "1501", "1502", "1503", "1504", "1505", "1506", "1507", "1508", "1509", "1510"
        "1511", "1512", "1513", "1514", "1515", "1516", "1517"
    ],
    "Fruits": [
        "0709", "0801", "0802", "0803", "0804", "0805", "0806", "0807", "0808", "0809", 
        "0810", "0813", "2009"
    ],
    "Meat and Prep": [
        "0201", "0203", "0204", "0205", "0207", "0208", "0209", "2301"
    ],
    "Processed Cereals": [
        "1006", "1101", "1102", "1104", "1105", "1106", "1107", "1108", "1109", "1302",
        "1602", "1901", "1902", "1903", "1904", "1905", "2005", "2007", "2104", "2106", 
        "3505"
    ],
    "Sugar and Honey": [
        "1701", "1702", "1703", "1704", "2006"
    ],
    "Unprocessed Cereals": [
        "1001", "1002", "1003", "1004", "1005", "1007", "1008"
    ],
    "Vegetables": [
        "0701", "0702", "0703", "0704", "0705", "0706", "0707", "0708", "0709", "0710",
        "0712", "0713", "0714", "1209", "2009"
    ]
}

## 3.2 Identify 'Food Security' groups of interest via CPC

In [9]:
impact_im["food_group"] = np.nan

for food_group in list(hsa4_codes_food.keys()):
    impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group

display(impact_im.head(2))

/var/folders/_k/81z8ryjd28lc4f0bb4dm7gkw0000gn/T/ipykernel_27848/1412186970.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Dairy' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl,food_group
0,0,882,Samoa,554,New Zealand,Import,Sea,40690,040690 | Dairy produce; chees,0406,0406 | Cheese and curd,04,04 | Dairy produce; birds,2295,Cheese and curd,Agriculture,22,MIL - Dairy products,39813.073551,NaN,39813.073551,kg,9178.55908,9178.55908,8153.717463,2813.0,1747.916623,14252.018293,['Container'],['Container'],43.833339,43.748538,43.756967,43.756967,43.798400,43.817147,45.440501,75.782749,48.460043,45.654378,45.801012,47.236954,44.617395,73.542473,50.724276,47.515919,46.615835,48.467961,43.899522,81.151083,60.833487,55.414975,51.897461,50.241822,43.196277,87.826141,70.870445,66.644570,54.813267,53.401706,42.592554,89.466089,84.610828,87.042609,59.057668,55.859505,41.739253,85.511644,82.867310,88.183568,69.791232,56.569521,41.055409,77.682360,80.210559,87.531271,68.394472,57.188444,40.362654,71.278985,73.045706,84.404435,69.641151,57.623039,2295,Dairy
1,1,882,Samoa,554,New Zealand,Import,Sea,50400,050400 | Animal products; gut,0504,"0504 | Guts, bladders and s",05,05 | Animal originated pr,3911,"Raw offal, inedible (including pigs' bristles,...",Agriculture,10,OAP - Animal products n.e.c.,4805.128484,NaN,4805.128484,kg,99.80000,99.80000,1130.646732,2813.0,1747.916623,1976.276218,['Container'],['Container'],6.078212,6.066453,6.067622,6.067622,6.073367,6.075967,6.301071,10.508522,6.719780,6.330729,6.351062,6.550179,6.186934,10.197871,7.033753,6.588862,6.464051,6.720878,6.087389,11.252929,8.435561,7.684196,7.196435,6.966853,5.989873,12.178536,9.827350,9.241363,7.600759,7.405023,5.906157,12.405941,11.732680,12.069886,8.189315,7.745837,5.787832,11.857593,11.490912,12.228099,9.677700,7.844292,5.693006,10.771934,11.122510,12.137647,9.484016,7.930116,5.596944,9.884001,10.128986,11.704060,9.656888,7.990380,3911,NaN


## 3.3 Inspect the output

In [10]:
impact_im_cpc_food = impact_im[(~impact_im.food_group.isna())]
impact_im_cpc_food_g = impact_im_cpc_food.groupby("food_group").agg({"HS6_desc":"count", "cifvalue":"sum", "vol_kg_vtw":"sum"}).reset_index().sort_values(by="cifvalue", ascending=False).round(2).rename(columns={"HS6_desc":"N_com"}).reset_index(drop=True)
impact_im_cpc_food_g["pct_val"] = round(100 * impact_im_cpc_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_cpc_food_g["pct_vol"] = round(100 * impact_im_cpc_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Proportion of imports identified as food security-relevant:\n")
print("Value: ${0}m of ${1}m ({2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / 1e6, 1),
    round(impact_im.cifvalue.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))
print("Volume: {0} of {1} million kilograms ({2}%)".format(
    round(impact_im_cpc_food.vol_kg_vtw.sum() / 1e6, 1),
    round(impact_im.vol_kg_vtw.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.vol_kg_vtw.sum() / impact_im.vol_kg_vtw.sum(), 1)
))
print("Value-to-Weight: {0} from {1} US$ per kg (+{2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum(), 2),
    round(impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum(), 2),
    round(100 * ((impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum()) - (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum())) / (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum()), 1)
))
print("Average distance to market (volume-weighted): {0} from {1} nm (+{2}%)".format(
    round((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000), 1),
    round((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000), 1),
    round(100 * (((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000)) - ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000))) / \
          ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000)), 1)
))
print("\n{0}\n".format(impact_im_cpc_food_g.head(15).round(1)))

impact_im_cpc_food_g.to_csv(output_dir + "{0}/food_total_usd_{1}.csv".format(iso_country, version), index=False)

Proportion of imports identified as food security-relevant:

Value: $89.5m of $413.7m (21.6%)
Volume: 81.2 of 451.3 million kilograms (18.0%)
Value-to-Weight: 1.1 from 0.92 US$ per kg (+20.1%)
Average distance to market (volume-weighted): 3104.7 from 4301.1 nm (+-27.8%)

            food_group  N_com    cifvalue  vol_kg_vtw  pct_val  pct_vol
0    Processed Cereals    376  34727429.8  36814494.8      8.4      8.2
1        Meat and Prep     89  22017664.7  11124406.0      5.3      2.5
2                Dairy     90  15965353.5  11599991.2      3.9      2.6
3      Sugar and Honey     78   6097981.6  11032537.0      1.5      2.4
4           Vegetables    170   4843570.6   6643166.4      1.2      1.5
5        Fats and Oils     61   3871682.0   2152590.4      0.9      0.5
6               Fruits    105   1890205.5   1734772.1      0.5      0.4
7  Unprocessed Cereals     21     40989.0    146482.4      0.0      0.0



## 3.3 Derive results for 2029, 2041 and 2050

### 3.3.1 Total FEM costs by scenario (US$)

In [11]:
print("\nTotal FEM costs by scenario (US$)...\n")

d = {"food_group": list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["bau", "levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            round(100 * impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum(), 1) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_usd_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$)...



,food_group,bau_29,levy_29,asis_29,brazil_29,liscr_29,direct_29,bau_41,levy_41,asis_41,brazil_41,liscr_41,direct_41,bau_50,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,8245438.9,13751213.4,8793352.1,8284248.2,8310855.7,8571415.7,7728662.7,16234133.9,15353119.0,15794379.5,10716351.8,10136026.8,7324034.5,12933979.8,13254561.4,15315667.8,12636785.3,10456030.2
1,Fats and Oils,2894126.5,4826638.4,3086442.6,2907748.5,2917087.7,3008543.5,2712739.5,5698136.7,5388902.9,5543784.1,3761410.2,3557717.8,2570716.1,4539791.6,4652314.9,5375757.6,4435477.2,3670038.1
2,Fruits,1569623.6,2617717.4,1673925.8,1577011.5,1582076.5,1631677.4,1471248.7,3090372.8,2922660.4,3006659.9,2039993.1,1929520.9,1394222.6,2462146.7,2523173.4,2915531.1,2405572.0,1990437.6
3,Meat and Prep,16839142.2,28083239.9,17958111.0,16918400.1,16972738.9,17504864.2,15783762.6,33153952.5,31354711.1,32255869.8,21885332.6,20700171.2,14957415.9,26414255.0,27068958.7,31278226.9,25807313.4,21353694.1
4,Processed Cereals,37078931.9,61837861.4,39542844.1,37253453.8,37373105.2,38544817.8,34755039.8,73003311.8,69041474.1,71025779.4,48190385.7,45580720.9,32935466.6,58162841.7,59604465.9,68873059.7,56826387.3,47019745.0
5,Sugar and Honey,8839159.5,14741382.6,9426525.8,8880763.4,8909286.8,9188608.6,8285172.3,17403088.1,16458634.9,16931668.8,11487992.9,10865881.0,7851408.6,13865303.3,14208968.7,16418487.0,13546709.1,11208926.6
6,Unprocessed Cereals,120805.0,201470.8,128832.5,121373.6,121763.4,125580.9,113233.6,237848.4,224940.5,231405.5,157006.6,148504.2,107305.4,189497.4,194194.3,224391.8,185143.2,153192.6
7,Vegetables,4839788.6,8071488.7,5161394.8,4862568.4,4878186.1,5031125.8,4536458.8,9528877.5,9011752.1,9270756.8,6290129.4,5949498.7,4298956.0,7591800.7,7779971.1,8989769.5,7417357.8,6137329.6


### 3.3.2 Total FEM costs by scenario (% cif)

In [12]:
print("\nTotal FEM costs by scenario (% cif)...\n")

d = {"food_group":list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            np.round(100 * (impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum() - impact_im[(impact_im.food_group == hsa4_codes_food_key)]["bau_usd_{1}".format(scenario, year)].sum()) / \
                     impact_im[(impact_im.food_group == hsa4_codes_food_key)].cifvalue.sum(), 2) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_pct_cif_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (% cif)...



,food_group,levy_29,asis_29,brazil_29,liscr_29,direct_29,levy_41,asis_41,brazil_41,liscr_41,direct_41,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,0.34,0.03,0.00,0.00,0.02,0.53,0.48,0.51,0.19,0.15,0.35,0.37,0.50,0.33,0.20
1,Fats and Oils,0.50,0.05,0.00,0.01,0.03,0.77,0.69,0.73,0.27,0.22,0.51,0.54,0.72,0.48,0.28
2,Fruits,0.55,0.06,0.00,0.01,0.03,0.86,0.77,0.81,0.30,0.24,0.56,0.60,0.80,0.54,0.32
3,Meat and Prep,0.51,0.05,0.00,0.01,0.03,0.79,0.71,0.75,0.28,0.22,0.52,0.55,0.74,0.49,0.29
4,Processed Cereals,0.71,0.07,0.01,0.01,0.04,1.10,0.99,1.04,0.39,0.31,0.73,0.77,1.03,0.69,0.41
5,Sugar and Honey,0.97,0.10,0.01,0.01,0.06,1.50,1.34,1.42,0.53,0.42,0.99,1.04,1.40,0.93,0.55
6,Unprocessed Cereals,1.97,0.20,0.01,0.02,0.12,3.04,2.73,2.88,1.07,0.86,2.01,2.12,2.86,1.90,1.12
7,Vegetables,0.67,0.07,0.00,0.01,0.04,1.03,0.92,0.98,0.36,0.29,0.68,0.72,0.97,0.64,0.38
